In [14]:
# импорт зависимосте

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import xarray as xr

project_root = Path.cwd().resolve()
while not (project_root / "userpwd.py").exists():
    project_root = project_root.parent
    assert project_root != project_root.parent, "Could not locate repo root with userpwd.py"

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import psycopg
import userpwd

In [15]:
def req(y0, y1):
    return """
                       select dt, speed, temperature, density
                       from rss.ace_at_earth_1h
                       where (
                       ace_at_earth_1h.dt >= DATE '{0}-01-01'
                       and
                       ace_at_earth_1h.dt <= DATE '{1}-01-01'
                       )
""".format(y0, y1)

In [16]:
years = list(range(2017, 2020))
years

[2017, 2018, 2019]

In [17]:
reqs = []

for i in range(len(years) - 1):
    reqs.append(req(years[i], years[i + 1]))
reqs

["\n                       select dt, speed, temperature, density\n                       from rss.ace_at_earth_1h\n                       where (\n                       ace_at_earth_1h.dt >= DATE '2017-01-01'\n                       and\n                       ace_at_earth_1h.dt <= DATE '2018-01-01'\n                       )\n",
 "\n                       select dt, speed, temperature, density\n                       from rss.ace_at_earth_1h\n                       where (\n                       ace_at_earth_1h.dt >= DATE '2018-01-01'\n                       and\n                       ace_at_earth_1h.dt <= DATE '2019-01-01'\n                       )\n"]

In [18]:
res = []

conn = psycopg.connect(
    host="213.131.1.41", user="selector", dbname="smdc", password=userpwd.userpwd_postgre
)

In [19]:
with conn.cursor() as cur:
    for req in reqs:
        print(req)
        for row in cur.execute(req):
            res.append(row)


                       select dt, speed, temperature, density
                       from rss.ace_at_earth_1h
                       where (
                       ace_at_earth_1h.dt >= DATE '2017-01-01'
                       and
                       ace_at_earth_1h.dt <= DATE '2018-01-01'
                       )


                       select dt, speed, temperature, density
                       from rss.ace_at_earth_1h
                       where (
                       ace_at_earth_1h.dt >= DATE '2018-01-01'
                       and
                       ace_at_earth_1h.dt <= DATE '2019-01-01'
                       )



In [20]:
df = pd.DataFrame(np.array(res), columns=["date", "speed", "temperature", "density"])

df["date"] = pd.to_datetime(df["date"])

df.set_index("date", inplace=True)

df

,speed,temperature,density
date,,,
2017-01-08 15:00:00,704.73,171658.333333,1.799167
2017-02-20 16:00:00,499.113333,136008.333333,1.8625
2017-01-07 09:00:00,673.856667,172925.0,1.753333
2017-02-06 22:00:00,550.2,67000.0,2.8
2017-02-13 17:00:00,330.054167,33604.166667,2.655833
...,...,...,...
2018-12-31 10:00:00,495.058475,107594.915254,3.369643
2018-12-31 13:00:00,469.189655,147939.655172,3.287319
2018-12-31 18:00:00,468.109322,117515.254237,1.24031


In [22]:
df.to_parquet("../Data/ACE At Earth 1h.parquet")